# 01 — MIDRC tiny download

**Goal of this notebook (Phase 0, intentionally tiny):**

1. Verify the local Gen3 client + credentials.
2. Connect to MIDRC with the Gen3 Python SDK.
3. Query a **tiny** number of `data_file` records for two broad classes:
   - `chest_xray`  (modality CR/DX, chest radiograph study descriptions)
   - `head_ct`     (modality CT, brain w/o contrast)
4. Save the metadata CSVs.
5. Download only 5–10 DICOMs per class via `gen3-client`.
6. Verify files exist on disk.

This notebook does **not** train anything. It does **not** download a large cohort.
If anything fails, it prints the exact command/error so we can debug.

Stakeholder reminder:
- Curtis Langlotz / Christopher Carr / George Shih — keep it open source, local, series-level, and start tiny.

## Section 0 — Paths / config

Repo paths are detected from the current working directory. The credentials path defaults to `~/Downloads/credentials.json`.

In [12]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

API          = "https://data.midrc.org"
CRED         = str(Path.home() / "Downloads" / "credentials.json")
GEN3_CLIENT  = "/Applications/gen3-client"
PROFILE      = "midrc"

REPO_ROOT    = Path.cwd().resolve()
if REPO_ROOT.name == "notebooks":
    REPO_ROOT = REPO_ROOT.parent
OUT_DIR      = REPO_ROOT / "data" / "raw"
META_DIR     = REPO_ROOT / "data" / "metadata"
MANIFEST_DIR = REPO_ROOT / "data" / "manifests"

for p in (OUT_DIR, META_DIR, MANIFEST_DIR):
    os.makedirs(p, exist_ok=True)
    print("ok:", p)

MAX_PER_CLASS = 5  # KEEP TINY. Increase only after a successful end-to-end run.

ok: /Users/lucaspu/HelperV1/data/raw
ok: /Users/lucaspu/HelperV1/data/metadata
ok: /Users/lucaspu/HelperV1/data/manifests


## Section 1 — Check credentials and gen3-client

If `auth` fails, the cell prints the **exact** `configure` command to run in a terminal.

In [13]:
def _exists(label, path):
    ok = Path(path).exists()
    print(f"[{'OK ' if ok else 'MISSING'}] {label}: {path}")
    return ok

cred_ok   = _exists("credentials.json", CRED)
client_ok = _exists("gen3-client", GEN3_CLIENT)

configure_cmd = (
    f"{GEN3_CLIENT} configure \\\n"
    f"  --profile={PROFILE} \\\n"
    f"  --cred={CRED} \\\n"
    f"  --apiendpoint={API}/"
)

if cred_ok and client_ok:
    try:
        res = subprocess.run(
            [GEN3_CLIENT, "auth", f"--profile={PROFILE}"],
            capture_output=True, text=True, timeout=30,
        )
        print("--- gen3-client auth stdout ---")
        print(res.stdout)
        if res.returncode != 0:
            print("--- gen3-client auth stderr ---")
            print(res.stderr)
            print("\nProfile may not be configured. Run this in a terminal:\n")
            print(configure_cmd)
    except Exception as exc:
        print("gen3-client auth failed:", exc)
        print("\nTry configuring the profile first:\n")
        print(configure_cmd)
else:
    print("\nFix the missing item(s) above, then run:\n")
    print(configure_cmd)

[OK ] credentials.json: /Users/lucaspu/Downloads/credentials.json
[OK ] gen3-client: /Applications/gen3-client
--- gen3-client auth stdout ---



## Section 2 — Connect with the Gen3 Python SDK

We use `Gen3Auth` (token refresh from `credentials.json`) and `Gen3Query` (the
GraphQL / flat-query wrapper). We do **not** assume specific field names —
if the query in Section 3 returns nothing, we print whatever columns we got.

In [14]:
try:
    from gen3.auth import Gen3Auth
    from gen3.query import Gen3Query
except ImportError as exc:
    print("gen3 SDK is not installed. Run:  pip install -r requirements.txt")
    raise

auth = Gen3Auth(API, refresh_file=CRED)
query = Gen3Query(auth)
print("Gen3Auth + Gen3Query initialized for", API)

Gen3Auth + Gen3Query initialized for https://data.midrc.org


## Section 3 — Query tiny metadata samples

We query the `data_file` node (this is what holds the GUID/object_id we'll
later hand to `gen3-client download-single`). We restrict by `data_format`,
`modality`, and a small whitelist of `study_description` strings drawn from
the MIDRC Explorer.

**MAX_PER_CLASS keeps the cohort small.** Do not raise it until everything
downstream works.

In [15]:
import pandas as pd

# Whitelists drawn from the messy free-text MIDRC Explorer values.
CHEST_XRAY_STUDY_DESCRIPTIONS = [
    "XR CHEST 1 VIEW AP",
    "XR CHEST 1 VW, FRONTAL",
    "XR Chest AP or PA",
    "XR PORT CHEST 1V",
]
HEAD_CT_STUDY_DESCRIPTIONS = [
    "BRAIN W/O CONTRAST (CT)-CS",
]

# Fields we'd like back. MIDRC's data_file schema varies; the SDK will return
# whatever it has and ignore unknown fields.
RETURN_FIELDS = [
    "object_id",
    "file_name",
    "file_size",
    "data_format",
    "data_type",
    "data_category",
    "modality",
    "study_description",
    "series_description",
    "body_part_examined",
    "study_uid",
    "series_uid",
    "case_ids",
]

def query_data_file(filters: dict, first: int) -> list[dict]:
    """Tiny wrapper around Gen3Query.query / raw_data_download.

    The MIDRC commons exposes a `data_file` index for the flat query API.
    We try `raw_data_download` first (returns plain JSON records, no GraphQL),
    and fall back to `query` if that method is unavailable in your SDK version.
    """
    # Method 1: raw_data_download (modern Gen3 SDK)
    if hasattr(query, "raw_data_download"):
        try:
            records = query.raw_data_download(
                data_type="data_file",
                fields=RETURN_FIELDS,
                filter_object=filters,
                first=first,
            )
            return records or []
        except Exception as exc:
            print("raw_data_download failed, will try query() fallback:", exc)

    # Method 2: GraphQL-style query() fallback.
    fields_str = " ".join(RETURN_FIELDS)
    # NOTE: This is a best-effort GraphQL string; if it fails, print so we can adapt.
    gql = f"""{{
      data_file(first: {first}, filter: {json.dumps(filters)}) {{
        {fields_str}
      }}
    }}"""
    try:
        res = query.query(gql)
        return (res.get("data", {}) or {}).get("data_file", []) or []
    except Exception as exc:
        print("query() fallback also failed:", exc)
        return []

def build_filter(modalities: list[str], descriptions: list[str]) -> dict:
    """Build the JSON filter object Gen3 expects (AND of two IN clauses)."""
    return {
        "AND": [
            {"IN": {"modality": modalities}},
            {"IN": {"study_description": descriptions}},
        ]
    }

def records_to_csv(records: list[dict], label: str) -> pd.DataFrame:
    df = pd.DataFrame(records)
    out = META_DIR / f"{label}_records.csv"
    df.to_csv(out, index=False)
    print(f"[{label}] saved {len(df)} rows -> {out}")
    if not df.empty:
        print(f"[{label}] columns: {list(df.columns)}")
        if "object_id" not in df.columns:
            print(f"[{label}] WARNING: no 'object_id' column. Inspect df.columns above.")
    else:
        print(f"[{label}] returned 0 records. Try widening filters or check auth.")
    return df

chest_filter = build_filter(["CR", "DX"], CHEST_XRAY_STUDY_DESCRIPTIONS)
head_filter  = build_filter(["CT"],       HEAD_CT_STUDY_DESCRIPTIONS)

print("chest_xray filter:", json.dumps(chest_filter, indent=2))
print("head_ct    filter:", json.dumps(head_filter,  indent=2))

chest_records = query_data_file(chest_filter, first=MAX_PER_CLASS)
head_records  = query_data_file(head_filter,  first=MAX_PER_CLASS)

chest_df = records_to_csv(chest_records, "chest_xray")
head_df  = records_to_csv(head_records,  "head_ct")

chest_xray filter: {
  "AND": [
    {
      "IN": {
        "modality": [
          "CR",
          "DX"
        ]
      }
    },
    {
      "IN": {
        "study_description": [
          "XR CHEST 1 VIEW AP",
          "XR CHEST 1 VW, FRONTAL",
          "XR Chest AP or PA",
          "XR PORT CHEST 1V"
        ]
      }
    }
  ]
}
head_ct    filter: {
  "AND": [
    {
      "IN": {
        "modality": [
          "CT"
        ]
      }
    },
    {
      "IN": {
        "study_description": [
          "BRAIN W/O CONTRAST (CT)-CS"
        ]
      }
    }
  ]
}
[chest_xray] saved 5 rows -> /Users/lucaspu/HelperV1/data/metadata/chest_xray_records.csv
[chest_xray] columns: ['data_format', 'modality', 'study_uid', 'case_ids', 'series_uid', 'file_name', 'study_description', 'data_category', 'series_description', 'body_part_examined', 'object_id', 'file_size', 'data_type']
[head_ct] saved 5 rows -> /Users/lucaspu/HelperV1/data/metadata/head_ct_records.csv
[head_ct] columns: ['data_form

## Section 4 — Download tiny files via `gen3-client download-single`

For each `object_id`, we shell out to `gen3-client`. We print every command
we run. If the exact flag syntax differs from the installed client version,
we print `--help` so you can adapt.

In [16]:
def gen3_help() -> None:
    """Print download-single help to debug flag differences."""
    try:
        res = subprocess.run(
            [GEN3_CLIENT, "download-single", "--help"],
            capture_output=True, text=True, timeout=15,
        )
        print(res.stdout or res.stderr)
    except Exception as exc:
        print("Could not run gen3-client download-single --help:", exc)

def download_one(object_id: str, class_dir: Path) -> bool:
    class_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        GEN3_CLIENT, "download-single",
        f"--profile={PROFILE}",
        f"--guid={object_id}",
        f"--download-path={class_dir}",
        "--no-prompt",
    ]
    print("$", " ".join(cmd))
    try:
        res = subprocess.run(cmd, capture_output=True, text=True, timeout=300)
    except Exception as exc:
        print("  subprocess error:", exc)
        return False
    if res.returncode != 0:
        print("  FAILED rc=", res.returncode)
        print("  stdout:", res.stdout)
        print("  stderr:", res.stderr)
        return False
    print("  ok")
    return True

def download_class(df: pd.DataFrame, class_name: str) -> None:
    class_dir = OUT_DIR / class_name
    if df.empty or "object_id" not in df.columns:
        print(f"[{class_name}] skipping: no object_id column / empty df.")
        return
    guids = [g for g in df["object_id"].dropna().tolist() if g]
    guids = guids[:MAX_PER_CLASS]
    print(f"[{class_name}] downloading {len(guids)} files -> {class_dir}")
    n_ok = 0
    for guid in guids:
        if download_one(guid, class_dir):
            n_ok += 1
    print(f"[{class_name}] done. {n_ok}/{len(guids)} succeeded.")

# Print the help once up front so the user can see the supported flags.
print("--- gen3-client download-single --help ---")
gen3_help()
print("------------------------------------------\n")

download_class(chest_df, "chest_xray")
download_class(head_df,  "head_ct")

--- gen3-client download-single --help ---
Gets a presigned URL for a file from a GUID and then downloads the specified file.

Usage:
  gen3-client download-single [flags]

Examples:
./gen3-client download-single --profile=<profile-name> --guid=206dfaa6-bcf1-4bc9-b2d0-77179f0f48fc

Flags:
      --download-path string     The directory in which to store the downloaded files (default ".")
      --filename-format string   The format of filename to be used, including "original", "guid" and "combined" (default "original")
      --guid string              Specify the guid for the data you would like to work with
  -h, --help                     help for download-single
      --no-prompt                If set to true, will not display user prompt message for confirmation
      --protocol string          Specify the preferred protocol with --protocol=gs
      --rename                   Only useful when "--filename-format=original", will rename file by appending a counter value to its filename 

## Section 5 — Verify downloads

Count `.dcm` files and any other downloaded files (extensions vary).
Print first few paths and total size on disk.

In [17]:
def summarize_dir(root: Path) -> None:
    if not root.exists():
        print(f"{root} does not exist.")
        return
    all_files = [p for p in root.rglob("*") if p.is_file()]
    dcm_files = [p for p in all_files if p.suffix.lower() in {".dcm", ".dicom"}]
    total_bytes = sum(p.stat().st_size for p in all_files)
    print(f"--- {root} ---")
    print(f"  total files  : {len(all_files)}")
    print(f"  .dcm/.dicom  : {len(dcm_files)}")
    print(f"  total size   : {total_bytes/1024/1024:.2f} MB")
    for p in all_files[:5]:
        print("   ", p)

summarize_dir(OUT_DIR / "chest_xray")
summarize_dir(OUT_DIR / "head_ct")
summarize_dir(OUT_DIR)

--- /Users/lucaspu/HelperV1/data/raw/chest_xray ---
  total files  : 5
  .dcm/.dicom  : 0
  total size   : 85.17 MB
    /Users/lucaspu/HelperV1/data/raw/chest_xray/419639-004486/1.2.826.0.1.3680043.10.474.419639.270396332866013633460836136843/1.2.826.0.1.3680043.10.474.419639.567892396369263363143494437150.zip
    /Users/lucaspu/HelperV1/data/raw/chest_xray/514382-011600/1.2.826.0.1.3680043.10.474.514382.1826468/1.2.826.0.1.3680043.10.474.514382.1826469.zip
    /Users/lucaspu/HelperV1/data/raw/chest_xray/10003752-w3uhjBebLUeBwtJdDSIA/2.16.840.1.114274.1818.54862041639390405042181213605039563172/2.16.840.1.114274.1818.54492409217557237789954425251006210218.zip
    /Users/lucaspu/HelperV1/data/raw/chest_xray/232451-001906/1.2.826.0.1.3680043.10.474.232451.68031/1.2.826.0.1.3680043.10.474.232451.68034.zip
    /Users/lucaspu/HelperV1/data/raw/chest_xray/10003752-1azc4d4F9UOj1vIbA3G0BQ/2.16.840.1.114274.1818.556972516376746133513059802808506623660/2.16.840.1.114274.1818.55351905090391027331

### Done.

Success criteria for this notebook:

1. `data/metadata/chest_xray_records.csv` and/or `data/metadata/head_ct_records.csv` exist.
2. A **nonzero** number of files exist under `data/raw/`. Even 1 file is a win.
3. If downloads failed, the printed command/error is enough to debug.

Next: run `02_inspect_dicoms.ipynb` (or `scripts/inspect_dicoms.py`) to group
downloaded files by `SeriesInstanceUID`.